# Classificação climática de Köppen-Geiger — CHELSA V2.1, normal 1991-2020

Versão em Python (API do Earth Engine) do script JS original (metodologia Alvares et al. 2013,
31 classes), adaptada para usar a normal do CHELSA (`tas` e `pr`; 12 bandas mensais cada) no lugar
do TerraClimate. `tas` já é a temperatura média mensal (não precisa de `(tmmx+tmmn)/2` como no
TerraClimate), e `pet` não entra nessa classificação. O cálculo está em [koppen_gee.py](koppen_gee.py).

Ao contrário do script original, esta versão classifica o Brasil inteiro pixel a pixel na resolução
nativa do CHELSA (~928 m), sem amostragem por município: não há tabela por município nem exportação
para o Google Drive. A exportação (seção 6) gera um GeoTIFF local, no mesmo padrão do notebook de
Holdridge.

**Correções em relação ao script JS** (ver [koppen_gee.py](koppen_gee.py)): a sazonalidade dos climas C
(f/s/w) segue Kottek et al. (2006) -- f é "nem s nem w"; antes, pixels com mês mais seco < 40 mm sem seca
sazonal forte ficavam sem classe (~5% da área C do Brasil). E verão e inverno são trocados ao norte do
equador (As/Aw, limiar do grupo B e s/w em Roraima e no Amapá).

## 1. Configuração

In [1]:
import sys
sys.path.insert(0, ".")

import ee
import geemap
import pandas as pd
import koppen_gee as k

PROJETO = "fcoliveira"

# Assets separados por variavel (12 bandas mensais cada; ajuste para o caminho onde voce subiu as imagens)
ASSET_TAS = "projects/fcoliveira/assets/chelsa_brasil_tas_normal_1991_2020"
ASSET_PR = "projects/fcoliveira/assets/chelsa_brasil_pr_normal_1991_2020"

# Asset de saida com a classificacao (referencia; a exportacao e manual, ver secao 6)
ASSET_SAIDA = "projects/fcoliveira/assets/CHELSA/Koppen_CHELSA_BR_1991_2020"


## 2. Earth Engine e região (Brasil)

In [2]:
try:
    ee.Initialize(project=PROJETO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO)

# 'brasil' usa o contorno real do pais (FAO GAUL) para clip/geometria/mapa.
brasil = (ee.FeatureCollection("FAO/GAUL/2015/level0")
          .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))
estados = (ee.FeatureCollection("FAO/GAUL/2015/level1")
           .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
/Users/coted-sh/Python/MapBiomas_analises/climas/chelsa_climas_brasil/.venv/lib/python3.13/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


## 3. Carregar a normal e classificar

In [3]:
normal = k.carregar_normal(ASSET_TAS, ASSET_PR)
print("Bandas:", normal.bandNames().getInfo())

klass = k.classificar_koppen(normal).clip(brasil)


Bandas: ['tas_01', 'tas_02', 'tas_03', 'tas_04', 'tas_05', 'tas_06', 'tas_07', 'tas_08', 'tas_09', 'tas_10', 'tas_11', 'tas_12', 'pr_01', 'pr_02', 'pr_03', 'pr_04', 'pr_05', 'pr_06', 'pr_07', 'pr_08', 'pr_09', 'pr_10', 'pr_11', 'pr_12']


## 4. Verificação: área por classe

No Brasil não se esperam classes D (continental) ou E (polar) -- fisicamente incompatíveis com as
latitudes/altitudes do país. Se aparecerem com área relevante, é sinal de erro na classificação
(ex.: unidades trocadas).

In [4]:
hist = klass.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=brasil.geometry(),
    scale=5000,
    maxPixels=1e13,
    bestEffort=True,
).get("koppen").getInfo()

tabela = (pd.Series(hist, name="pixels_5km").rename_axis("classe_id").reset_index()
          .assign(classe_id=lambda d: d.classe_id.astype(int))
          .sort_values("pixels_5km", ascending=False).reset_index(drop=True))
tabela["classe"] = tabela["classe_id"].map(k.LEGENDA)
tabela["area_km2"] = tabela["pixels_5km"] * 25
tabela["pct"] = (100 * tabela["pixels_5km"] / tabela["pixels_5km"].sum()).round(2)
display(tabela[["classe_id", "classe", "area_km2", "pct"]])

pct_d_e = tabela.loc[tabela["classe_id"].between(18, 31), "pct"].sum()
if pct_d_e > 0:
    print(f"ATENCAO: {pct_d_e:.2f}% do Brasil caiu em classes D/E (continental/polar). Reveja a classificacao.")


,classe_id,classe,area_km2,pct
0,4,Aw,4.120885e+06,46.96
1,2,Am,1.729207e+06,19.71
2,1,Af,1.377975e+06,15.70
3,9,Cfa,5.380236e+05,6.13
4,5,BSh,5.141561e+05,5.86
5,10,Cfb,1.768565e+05,2.02
6,3,As,1.575529e+05,1.80
7,15,Cwa,9.800000e+04,1.12
8,16,Cwb,5.262500e+04,0.60
9,7,BWh,9.500000e+03,0.11


## 5. Mapa

In [5]:
vis = {"min": 1, "max": 31, "palette": k.PALETA}

Map = geemap.Map(center=[-14, -52], zoom=4)
Map.addLayer(klass, vis, "Köppen-Geiger (31 classes)")
Map.addLayer(estados.style(color="000000", fillColor="00000000", width=1), {}, "Estados")
Map.addLayer(brasil.style(color="000000", fillColor="00000000", width=2), {}, "Brasil")

legenda_mapa = {f"{cod} {nome}": k.PALETA[cod - 1].lstrip("#") for cod, nome in k.LEGENDA.items()}
Map.add_legend(title="Köppen-Geiger", legend_dict=legenda_mapa)
Map


Map(center=[-14, -52], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

## 6. Exportação

A célula abaixo baixa `klass` como GeoTIFF local, na resolução nativa do CHELSA (~928 m, a mesma dos
assets de entrada) e recortado por `brasil`. O arquivo sai em `climas/dados_chelsa/koppen/`, no mesmo
padrão do notebook de Holdridge: gera local e depois sobe manualmente para o GEE (via bucket do GCS,
igual ao fluxo em `chelsa_brasil.ipynb`).

In [6]:
from pathlib import Path

SAIDA_DIR = Path("../dados_chelsa/koppen")
SAIDA_DIR.mkdir(parents=True, exist_ok=True)
caminho_local = SAIDA_DIR / "Koppen_CHELSA_BR_1991_2020.tif"

proj = normal.select(0).projection()

# download_ee_image (nao ee_export_image) porque a imagem passa do limite de 48MB de
# download direto do GEE; ele baixa em tiles e remonta um unico GeoTIFF.
geemap.download_ee_image(
    klass.toByte().rename("koppen"),
    filename=str(caminho_local),
    region=brasil.geometry(),
    crs=proj.crs().getInfo(),
    scale=proj.nominalScale().getInfo(),
    dtype="uint8",
)

print(f"Salvo em: {caminho_local.resolve()}")
print("\nDepois, subir para um bucket no GCS e criar o asset (mesmo fluxo do chelsa_brasil.ipynb):")
print(f"  gsutil cp {caminho_local} gs://SEU_BUCKET/Koppen_CHELSA_BR_1991_2020.tif")
print(f"  earthengine upload image --asset_id={ASSET_SAIDA} --pyramiding_policy=mode "
      "gs://SEU_BUCKET/Koppen_CHELSA_BR_1991_2020.tif")


  0%|          |0/20 tiles [00:00<?]

Salvo em: /Users/coted-sh/Python/MapBiomas_analises/climas/dados_chelsa/koppen/Koppen_CHELSA_BR_1991_2020.tif

Depois, subir para um bucket no GCS e criar o asset (mesmo fluxo do chelsa_brasil.ipynb):
  gsutil cp ../dados_chelsa/koppen/Koppen_CHELSA_BR_1991_2020.tif gs://SEU_BUCKET/Koppen_CHELSA_BR_1991_2020.tif
  earthengine upload image --asset_id=projects/fcoliveira/assets/CHELSA/Koppen_CHELSA_BR_1991_2020 --pyramiding_policy=mode gs://SEU_BUCKET/Koppen_CHELSA_BR_1991_2020.tif
